In [1]:
import pandas as pd

def transcript_to_dataframe(transcript_file):
    with open(transcript_file, 'r', encoding='utf8') as file:
        lines = file.readlines()

    data = []
    for line in lines:
        if line.strip():  # Skip empty lines
            speaker, text = line.split(":", 1)
            data.append({
                "Speaker": speaker.strip(),
                "Text": text.strip()
            })

    df = pd.DataFrame(data)

    return df

In [2]:
import os

def caps_check(base_directory):
    result = []
    for root, dirs, files in os.walk(base_directory):
        for file in files:
            if file.startswith("clean"):
                with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                for i, line in enumerate(lines, start=1):
                    # Remove the speaker's name
                    text = line.split(':', 1)[1] if ':' in line else line
                    words = text.split()
                    # Check if there are 4 or more consecutive capitalized words
                    for j in range(len(words) - 3):  
                        if words[j].isupper() and words[j+1].isupper() and words[j+2].isupper() and words[j+3].isupper():
                            result.append({
                                'File': file,
                                'Line': i,
                                'Text': text.strip()
                            })
                            break  # No need to check the rest of the line

    return pd.DataFrame(result)

df = caps_check("all_debates")

In [ ]:
import re

def add_original_sentence(df, base_directory):
    df['Original Text'] = ''  # Add a new column for the original sentences
    df['Filename'] = ''  # Add a new column for the filename
    df['Line Number'] = ''  # Add a new column for the line number
    
    for index, row in df.iterrows():
        # Construct the file path to the raw file
        raw_file = row['File'].replace("clean-", "", 1).replace(".txt", "-raw.txt", 1)
        raw_file_year = raw_file.split('-')[0]  # Extract the year from the filename
        raw_file_path = os.path.join(base_directory, raw_file_year, 'raw', raw_file)

        # Open the raw file and search for the original sentence
        with open(raw_file_path, 'r', encoding='utf-8') as file:
            raw_text = file.read()
            lines = raw_text.split('\n')  # Split the text into lines

        # Create a pattern with the first 10 words of the sentence (to avoid false positives)
        pattern = ' '.join(row['Text'].split()[:10])
        for line_number, line in enumerate(lines, start=1):
            match = re.search(rf"({pattern}[^\.]*\.)", line, re.IGNORECASE)
            if match:
                df.loc[index, 'Original Text'] = match.group(1)
                df.loc[index, 'Filename'] = raw_file  # Add the filename
                df.loc[index, 'Line Number'] = line_number  # Add the line number
                break  # No need to check the rest of the lines once a match is found

    # Sort the DataFrame
    df.sort_values(by='Original Text', ascending=False, na_position='first', inplace=True)

    return df

# Call the function on your DataFrame
df = add_original_sentence(df, 'all_debates').sort_values("File")
pd.set_option('display.max_colwidth', 0)
df

In [4]:
df[0:10]

,File,Line,Text,Original Text,Filename,Line Number
66,clean-1984-10-21-debate.txt,84,"But we were succeeding to the point that the Lebanese Government had been organized — if you will remember, there were the meetings in Geneva in which they began to meet with the hostile factional forces and try to put together some kind of a peace plan. WE WERE SUCCEEDING, AND THAT WAS WHY THE TERRORIST ACTS BEGAN. THERE ARE FORCES THERE — AND THAT INCLUDES SYRIA, IN MY MIND — WHO DON’T WANT US TO SUCCEED, WHO DON’T WANT THAT KIND OF A PEACE WITH A DOMINANT LEBANON, DOMINANT OVER ITS OWN TERRITORY. AND SO, THE TERRORIST ACTS BEGAN AND LED TO THE ONE GREAT TRAGEDY WHEN THEY WERE KILLED IN THAT SUICIDE BOMBING OF THE BUILDING. THEN THE MULTILATERAL FORCE WITHDREW FOR ONLY ONE REASON: We withdrew because we were no longer able to carry out the mission for which we had been sent in. But we went in in the interest of peace and to keep Israel and Syria from getting into the sixth war between them. And I have no apologies for our going on a peace mission.","But we were succeeding to the point that the Lebanese Government had been organized — if you will remember, there were the meetings in Geneva in which they began to meet with the hostile factional forces and try to put together some kind of a peace plan.",1984-10-21-debate-raw.txt,92
67,clean-1984-10-21-debate.txt,152,"NOW, WITH REGARD TO HAVING TO SAY WHETHER WE WOULD TRY TO SURVIVE IN THE EVENT OF A NUCLEAR WAR, OF COURSE WE WOULD. BUT LET ME ALSO POINT OUT THAT TO SEVERAL PARLIAMENTS AROUND THE WORLD, IN EUROPE AND IN ASIA, I HAVE MADE A STATEMENT TO EACH ONE OF THEM, AND I’LL REPEAT IT HERE: A nuclear war cannot be won and must never be fought. And that is why we are maintaining a deterrent and trying to achieve a deterrent capacity to where no one would believe that they could start such a war and escape with limited damage.","Now, with regard to having to say whether we would try to survive in the event of a nuclear war, of course we would.",1984-10-21-debate-raw.txt,161
22,clean-1992-10-11-debate.txt,153,"2ND THING, THERE ARE NOT MANY TIMES IN YOUR LIFE WHEN YOU GET TO TALK TO A WHOLE COUNTRY. BUT LET ME JUST SAY TO ALL OF AMERICA: if you hate people, I don’t want your vote. That’s how strongly I feel about it. (APPLAUSE)","2nd thing, there are not many times in your life when you get to talk to a whole country.",1992-10-11-debate-raw.txt,79
23,clean-1992-10-11-debate.txt,192,"MR. BUSH IS TRYING TO RUN AGAINST LYNDON JOHNSON AND JIMMY CARTER AND EVERYBODY IN THE WORLD BUT ME IN THIS RACE. I HAVE PROPOSED A MANAGED COMPETITION PLAN FOR HEALTH CARE. I WILL SAY AGAIN: you cannot control health care costs simply by cutting Medicare. Look what’s happened. The federal government has cut Medicare and Medicaid in the last few years, states have cut Medicaid — we’ve done it in Arkansas under budget pressures. But what happens? More and more people get on the rolls as poverty increases. If you don’t control the health care costs of the entire system, you cannot get control of it.",Mr. Bush is trying to run against Lyndon Johnson and Jimmy Carter and everybody in the world but me in this race.,1992-10-11-debate-raw.txt,128
24,clean-1992-10-13-debate.txt,19,"IN ST. LOUIS, MISSOURI, IN JUNE OF THIS YEAR, BILL CLINTON SAID THIS: “America is the mockery of the world.” He is wrong.","In St. Louis, Missouri, in June of this year, Bill Clinton said this: “America is the mockery of the world.",1992-10-13-debate-raw.txt,32
25,clean-1992-10-13-debate.txt,142,"YES, WE ARE MAKING A CONVERSION AND WE CAN GO TO A CIVIL SPACE RATHER THAN HAVING DEFENSE — OR THE DEFENSE INDUSTRY. WELL, LET ME SAY THIS: we would not have won the Cold War if we had listened to Senator Gore and his crowd, and had supported a nuclear freeze. If you would have supported that attitude — if you would have supported that attitude, we would not have won the Cold War. We won the Cold War because we invested and we went forward. (APPLAUSE.)",

In [5]:
print(df[df['Text'].str.contains('FORMER GOVERNOR RONALD REAGAN')])


Empty DataFrame
Columns: [File, Line, Text, Original Text, Filename, Line Number]
Index: []
